# Transformer from scratch: annotated walkthrough
This notebook builds each component step by step, with sanity checks and visualisations.
**The canonical, tested implementation lives in [`transformer/`](../transformer)**, and the reported results come from `notebooks/run_experiments.ipynb`.

*Revision note:* this version fixes several bugs in the first draft: 5-D padding masks in MHA, float masks that broke `&`, a LayerNorm that didn't match the standard formula, a residual-gradient experiment whose probe had an identically-zero gradient, a redundant final LayerNorm in Post-LN stacks, and an inverted description of positional-encoding frequencies. Outputs were cleared; re-run to regenerate them.

## 0. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import math

torch.manual_seed(42)
np.random.seed(42)

## 1. Embeddings and positional encoding

In [ ]:
class PositionalEncoding(nn.Module):
  def __init__(self, d_model, max_len=5000, dropout=0.1):
    super().__init__()
    self.dropout = nn.Dropout(p=dropout)
    pe = torch.zeros(max_len, d_model) # empty matrix of shape (max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0)/ d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)

    pe = pe.unsqueeze(0)
    self.register_buffer('pe', pe)

  def forward(self, x):
    x = x + self.pe[:, :x.size(1), :]
    return self.dropout(x)

print("PositionalEncoding class implemented!")
print("\nKEY POINTS TO REMEMBER:")
print("1. We PRE-COMPUTE positional encodings (they don't change during training)")
print("2. We use register_buffer (not nn.Parameter) - these aren't learned")
print("3. We add PE to embeddings, not concatenate")
print("4. Dropout is applied AFTER adding positional encoding")

In [ ]:
# Test
d_model = 512
seq_len = 100
batch_size = 2

embeddings = torch.randn(batch_size, seq_len, d_model)
pos_encoder = PositionalEncoding(d_model, max_len=5000, dropout=0.1)
output = pos_encoder(embeddings)

print(f"Input shape: {embeddings.shape}")
print(f"Output shape: {output.shape}")
print(f"Shapes match! (batch_size, seq_len, d_model)")

# Verify the encoding is added correctly
print(f"\nPositional encoding shape: {pos_encoder.pe.shape}")
print(f"Note: pe has batch dimension of 1, so it broadcasts to all batches")

In [ ]:
# Visualize Positional Encoding
d_model_viz = 128  # Smaller for easier visualization
max_len_viz = 100

pos_encoder_viz = PositionalEncoding(d_model_viz, max_len=max_len_viz, dropout=0.0)
pe_viz = pos_encoder_viz.pe.squeeze(0).numpy()


plt.figure(figsize=(15, 5))

plt.subplot(1, 2, 1)
plt.imshow(pe_viz.T, cmap='RdBu', aspect='auto', vmin=-1, vmax=1)
plt.colorbar()
plt.xlabel('Position in sequence')
plt.ylabel('Embedding dimension')
plt.title('Positional Encoding Heatmap')

# Individual dimensions over positions
plt.subplot(1, 2, 2)
for i in range(0, 8, 2):
    plt.plot(pe_viz[:50, i], label=f'Dim {i} (sin)')
    plt.plot(pe_viz[:50, i+1], '--', label=f'Dim {i+1} (cos)')

plt.xlabel('Position in sequence')
plt.ylabel('Encoding value')
plt.title('Positional Encoding Values\n(First 50 positions, 4 dim pairs)')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("\n🔍 WHAT TO NOTICE:")
print("1. Left plot: See the stripe patterns? Different frequencies for different dimensions!")
print("2. Right plot: Sine/cosine pairs create smooth, periodic patterns")
print("3. LOW dimension indices oscillate FAST (wavelength 2*pi at dim 0) - fine-grained position")
print("4. HIGH dimension indices oscillate SLOWLY (wavelength up to 10000*2*pi) - coarse position")

In [ ]:
# Token embedding
class TokenEmbedding(nn.Module):
  def __init__(self, vocab_size, d_model):
    super().__init__()
    self.embedding = nn.Embedding(vocab_size, d_model)
    self.d_model = d_model

  def forward(self, x):
    return self.embedding(x) * math.sqrt(self.d_model)

print("TokenEmbedding class implemented!")

In [ ]:
# Complete Embedding Module
class TransformerEmbedding(nn.Module):
    def __init__(self, vocab_size, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.token_embedding = TokenEmbedding(vocab_size, d_model)
        self.positional_encoding = PositionalEncoding(d_model, max_len, dropout)

    def forward(self, x):
        tok_emb = self.token_embedding(x)
        return self.positional_encoding(tok_emb)


print("TransformerEmbedding class implemented!")

In [ ]:
# Test
vocab_size = 1000
d_model = 512
batch_size = 2
seq_len = 10

# Create sample token indices
# These would normally come from tokenized text
tokens = torch.randint(0, vocab_size, (batch_size, seq_len))

print("Sample tokens (first sequence):", tokens[0].tolist())
print(f"Shape: {tokens.shape}\n")

# Create embedding module
embedding = TransformerEmbedding(vocab_size, d_model, max_len=5000, dropout=0.1)

# Get embeddings
output = embedding(tokens)

print(f"Output shape: {output.shape}")
print(f"Expected: (batch_size={batch_size}, seq_len={seq_len}, d_model={d_model})")
print(f"Success! Shape matches!")

## 2. Scaled dot-product attention

In [ ]:
# Scaled dot-product attention
def scaled_dot_product_attention(query, key, value, mask=None, dropout=None):
  d_k = query.size(-1)
  scores = torch.matmul(query, key.transpose(-2, -1))
  scores = scores / math.sqrt(d_k)

  if mask is not None:
        # Fill masked positions with -inf so softmax makes them ~0
        scores = scores.masked_fill(mask == 0, torch.finfo(scores.dtype).min)  # -1e9 overflows in fp16
  attention_weights = F.softmax(scores, dim=-1)


  if dropout is not None:
      attention_weights = dropout(attention_weights)

    # Multiply by values to get output
    # (batch, n_heads, seq_len_q, seq_len_k) × (batch, n_heads, seq_len_k, d_k)
    # → (batch, n_heads, seq_len_q, d_k)
  output = torch.matmul(attention_weights, value)

  return output, attention_weights
print("Scaled Dot-Product Attention implemented!")

In [ ]:
# Test attention
batch_size = 1
n_heads = 1
seq_len = 3
d_k = 4

# Create simple Q, K, V
torch.manual_seed(42)
query = torch.randn(batch_size, n_heads, seq_len, d_k)
key = torch.randn(batch_size, n_heads, seq_len, d_k)
value = torch.randn(batch_size, n_heads, seq_len, d_k)

print("Input shapes:")
print(f"  Query: {query.shape}")
print(f"  Key:   {key.shape}")
print(f"  Value: {value.shape}")

# Compute attention
output, attention_weights = scaled_dot_product_attention(query, key, value)

print(f"\nOutput shape: {output.shape}")
print(f"Attention weights shape: {attention_weights.shape}")

# Visualize attention weights
print("\n Attention Weights Matrix:")
print("(Each row shows how much position i attends to all positions)")
print(attention_weights[0, 0].detach().numpy())
print("- Each row sums to 1.0 (softmax property)")
print("- Values are between 0 and 1")
print("- These weights determine how much each position influences the output")

In [ ]:
# Visualise Attention
import seaborn as sns


seq_len = 10
query = torch.randn(1, 1, seq_len, d_k)
key = torch.randn(1, 1, seq_len, d_k)
value = torch.randn(1, 1, seq_len, d_k)

# Make the first position attend strongly to itself
query[0, 0, 0] = key[0, 0, 0] * 3

# Compute attention
output, attention_weights = scaled_dot_product_attention(query, key, value)


plt.figure(figsize=(8, 6))
sns.heatmap(attention_weights[0, 0].detach().numpy(),
            annot=True, fmt='.2f', cmap='YlOrRd',
            xticklabels=range(seq_len),
            yticklabels=range(seq_len))
plt.xlabel('Key Position (what we attend TO)')
plt.ylabel('Query Position (what is attending)')
plt.title('Attention Weights Heatmap\n(Brighter = More attention)')
plt.show()


print("- Each ROW is a query position")
print("- Each COLUMN is a key position")
print("- Value at (i,j) = how much position i attends to position j")
print("- Each row sums to 1.0")

## 3. Multi-head attention

In [ ]:
# Multihead attention
class MultiHeadAttention(nn.Module):
  def __init__(self, d_model, n_heads, dropout= 0.1):
    super().__init__()
    assert d_model % n_heads == 0, "d_model must be divisible by n_heads"

    self.d_model = d_model
    self.n_heads = n_heads
    self.d_k = d_model // n_heads  # Dimension per head

     # Linear projections for Q, K, V
    self.W_q = nn.Linear(d_model, d_model)  # Query projection
    self.W_k = nn.Linear(d_model, d_model)  # Key projection
    self.W_v = nn.Linear(d_model, d_model)  # Value projection

    # Output projection
    self.W_o = nn.Linear(d_model, d_model)

    self.dropout = nn.Dropout(dropout)

  def split_heads(self, x):
    batch_size, seq_len, _ = x.size()
    x = x.view(batch_size, seq_len, self.n_heads, self.d_k)
    return x.transpose(1, 2)

  def combine_heads(self, x):
    batch_size, n_heads, seq_len, d_k = x.size()

    x = x.transpose(1, 2)
    return x.contiguous().view(batch_size, seq_len, self.d_model)

  def forward(self, query, key, value, mask=None):
    batch_size = query.size(0)

    Q = self.W_q(query)  # (batch_size, seq_len_q, d_model)
    K = self.W_k(key)    # (batch_size, seq_len_k, d_model)
    V = self.W_v(value)  # (batch_size, seq_len_v, d_model)

    # Split into multiple heads
    Q = self.split_heads(Q)  # (batch_size, n_heads, seq_len_q, d_k)
    K = self.split_heads(K)  # (batch_size, n_heads, seq_len_k, d_k)
    V = self.split_heads(V)  # (batch_size, n_heads, seq_len_v, d_k)

    # Apply attention on all heads in parallel
    # Masks are passed in already broadcastable to (batch, heads, len_q, len_k).
    # Unsqueezing here would turn a (batch,1,1,len_k) padding mask into 5-D.

    attention_output, attention_weights = scaled_dot_product_attention(
            Q, K, V, mask, self.dropout
        )

    # Concatenate heads
    attention_output = self.combine_heads(attention_output)

    # Final linear projection
    output = self.W_o(attention_output)

    return output, attention_weights


print("Multi-Head Attention implemented!")

In [ ]:
# Test Multihead Attention
d_model = 512
n_heads = 8
seq_len = 10
batch_size = 2

mha = MultiHeadAttention(d_model, n_heads, dropout= 0.1)

x = torch.randn(batch_size, seq_len, d_model)
output, attention_weights = mha(x, x, x, mask=None)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("Attention weights shape:", attention_weights.shape)
print("\n Success! Output shape matches input shape")

# Verifying dimensions
print(f"\nVerifying dimensions:")
print(f"  d_model = {d_model}")
print(f"  n_heads = {n_heads}")
print(f"  d_k per head = {d_model // n_heads}")
print(f"  Attention has {n_heads} heads, each computing attention in {d_model // n_heads}D space")

In [ ]:
# Multi-Head Attention Visualisation

# slightly more interesting input pattern
seq_len = 15
x = torch.randn(1, seq_len, d_model)

# Forward pass
output, attention_weights = mha(x, x, x, mask=None)

# Visualizing first 4 heads
fig, axes = plt.subplots(2, 2, figsize=(12, 10))
axes = axes.flatten()

for i in range(4):
    ax = axes[i]
    # Extracting attention for head i
    attn = attention_weights[0, i].detach().numpy()

    sns.heatmap(attn, ax=ax, cmap='YlOrRd',
                xticklabels=False, yticklabels=False,
                cbar_kws={'label': 'Attention Weight'})
    ax.set_title(f'Head {i+1} Attention Pattern')
    ax.set_xlabel('Key Position')
    ax.set_ylabel('Query Position')

plt.suptitle('Untrained heads (random init): differences here are NOT learned - see trained maps later',
             fontsize=14, y=1.00)
plt.tight_layout()
plt.show()

## 4. Masking

In [ ]:
# Masking (Position i should only attend to positions <= i)
def look_ahead_mask(size):
    mask = torch.tril(torch.ones(size, size))
    return mask

seq_len = 10
mask = look_ahead_mask(seq_len)

plt.figure(figsize=(6, 5))
sns.heatmap(mask.numpy(), annot=True, fmt='.0f', cmap='RdYlGn',
            cbar=False, square=True)
plt.title('Look-Ahead Mask\n(1 = Can Attend, 0 = Masked)')
plt.xlabel('Key Position')
plt.ylabel('Query Position')
plt.show()

In [ ]:
# Test Attention With Masking

seq_len = 5
x = torch.randn(1, seq_len, d_model)


mask = look_ahead_mask(seq_len).bool()

# Adding batch dimension for compatibility
mask = mask.unsqueeze(0)  # (1, seq_len, seq_len)

# Apply attention WITH mask
output_masked, attn_masked = mha(x, x, x, mask=mask)

# Apply attention WITHOUT mask (for comparison)
output_unmasked, attn_unmasked = mha(x, x, x, mask=None)

# Visualize the difference
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Unmasked attention (first head)
sns.heatmap(attn_unmasked[0, 0].detach().numpy(), ax=axes[0],
            annot=True, fmt='.2f', cmap='YlOrRd')
axes[0].set_title('WITHOUT Mask\n(Can see future)')
axes[0].set_xlabel('Key Position')
axes[0].set_ylabel('Query Position')

# Masked attention (first head)
sns.heatmap(attn_masked[0, 0].detach().numpy(), ax=axes[1],
            annot=True, fmt='.2f', cmap='YlOrRd')
axes[1].set_title('WITH Mask\n(Cannot see future)')
axes[1].set_xlabel('Key Position')
axes[1].set_ylabel('Query Position')

plt.tight_layout()
plt.show()

## 5. Position-wise feed-forward network

In [ ]:
# FEED FORWARD NETWOK
class FeedForward(nn.Module):
  def __init__(self, d_model, d_ff, dropout= 0.1):
    super().__init__()

    # First linear layer (expand from d_model to d_ff)
    self.linear1 = nn.Linear(d_model, d_ff)

    # Second linear layer (project back from d_ff to d_model)
    self.linear2 = nn.Linear(d_ff, d_model)

    # Dropout for regularization
    self.dropout = nn.Dropout(dropout)

  def forward(self, x):
    # First linear transformation + ReLU
    # (batch_size, seq_len, d_model) → (batch_size, seq_len, d_ff)
    x = self.linear1(x)
    x = F.relu(x)

    # Apply dropout
    x = self.dropout(x)

    # Second linear transformation
    # (batch_size, seq_len, d_ff) → (batch_size, seq_len, d_model)
    x = self.linear2(x)

    return x

In [ ]:
# Test FEED FORWARD NETWORK
d_model = 512
d_ff = 2048
batch_size = 2
seq_len = 10


ffn = FeedForward(d_model, d_ff, dropout= 0.1)
x = torch.randn(batch_size, seq_len, d_model)

# Forward pass
output = ffn(x)

print("Input shape:", x.shape)
print("Output shape:", output.shape)
print("\n Input and output shapes match")

# Count parameters
total_params = sum(p.numel() for p in ffn.parameters())
print(f"\nTotal parameters in FFN: {total_params:,}")
print(f"  Linear1: {d_model} × {d_ff} = {d_model * d_ff:,}")
print(f"  Linear2: {d_ff} × {d_model} = {d_ff * d_model:,}")

In [ ]:
# Visualise FEED FORWARD NETWORK

# a simple input
x_simple = torch.randn(1, 5, d_model)

# Forward pass through first layer
ffn.eval()  # Set to eval mode to disable dropout
with torch.no_grad():
    h = ffn.linear1(x_simple)
    h_relu = F.relu(h)

    output = ffn.linear2(h_relu)

# Visualize activations
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Input
axes[0].imshow(x_simple[0].T.numpy(), aspect='auto', cmap='RdBu')
axes[0].set_title(f'Input\n(seq_len=5, d_model={d_model})')
axes[0].set_xlabel('Position')
axes[0].set_ylabel('Dimension')

# After ReLU (expanded)
axes[1].imshow(h_relu[0].T.numpy(), aspect='auto', cmap='RdBu')
axes[1].set_title(f'After Linear1 + ReLU\n(seq_len=5, d_ff={d_ff})')
axes[1].set_xlabel('Position')
axes[1].set_ylabel('Dimension')

# Output
axes[2].imshow(output[0].T.numpy(), aspect='auto', cmap='RdBu')
axes[2].set_title(f'Output\n(seq_len=5, d_model={d_model})')
axes[2].set_xlabel('Position')
axes[2].set_ylabel('Dimension')

plt.tight_layout()
plt.show()

## 6. Layer normalisation and residual connections

In [ ]:
# Layer Normalization

class LayerNorm(nn.Module):
    def __init__(self, d_model, eps=1e-5):
        super().__init__()

        # Learnable parameters
        self.gamma = nn.Parameter(torch.ones(d_model))   # Scale
        self.beta = nn.Parameter(torch.zeros(d_model))   # Shift

        self.eps = eps

    def forward(self, x):
        # Compute mean and std across the last dimension (d_model)
        # keepdim=True preserves the dimension for broadcasting
        mean = x.mean(dim=-1, keepdim=True)  # (batch_size, seq_len, 1)
        # BIASED variance, eps inside the sqrt -> identical to nn.LayerNorm.
        # (x.std() is the unbiased estimator and adding eps to std is a different formula.)
        var = x.var(dim=-1, keepdim=True, correction=0)  # (batch_size, seq_len, 1)

        x_norm = (x - mean) / torch.sqrt(var + self.eps)

        return self.gamma * x_norm + self.beta


assert torch.allclose(LayerNorm(64)(t := torch.randn(3, 5, 64)), nn.LayerNorm(64)(t), atol=1e-5)
print('LayerNorm matches nn.LayerNorm')

In [ ]:
# Residual Connection Wrapper (This makes it easy to add residuals to any layer)
class ResidualConnection(nn.Module):
    def __init__(self, d_model, dropout=0.1):
        super().__init__()
        self.norm = nn.LayerNorm(d_model)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x, sublayer):
        # Original paper (Post-LN):  LayerNorm(x + Dropout(Sublayer(x)))   <- what we use
        # Later variant (Pre-LN):    x + Dropout(Sublayer(LayerNorm(x)))
        return self.norm(x + self.dropout(sublayer(x)))

In [ ]:
# Test Layer Norm and Residuals
# Test Layer Normalization
d_model = 512
x = torch.randn(2, 10, d_model)

layer_norm = LayerNorm(d_model)
x_normalized = layer_norm(x)

print("Original stats:")
print(f"  Mean: {x.mean():.4f}")
print(f"  Std:  {x.std():.4f}")

print("\nAfter LayerNorm:")
print(f"  Mean: {x_normalized.mean():.4f} (should be ~0)")
print(f"  Std:  {x_normalized.std():.4f} (should be ~1)")


fig, axes = plt.subplots(1, 2, figsize=(12, 4))


axes[0].hist(x.flatten().numpy(), bins=50, alpha=0.7, color='blue')
axes[0].axvline(x.mean().item(), color='red', linestyle='--', label='Mean')
axes[0].set_title('Before Layer Normalization')
axes[0].set_xlabel('Value')
axes[0].set_ylabel('Frequency')
axes[0].legend()


axes[1].hist(x_normalized.flatten().detach().numpy(), bins=50, alpha=0.7, color='green')
axes[1].axvline(x_normalized.mean().item(), color='red', linestyle='--', label='Mean')
axes[1].set_title('After Layer Normalization')
axes[1].set_xlabel('Value')
axes[1].set_ylabel('Frequency')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
# Testing Residual Connection

#  a dummy sublayer (just FFN for testing)
ffn = FeedForward(d_model, d_ff=2048, dropout=0.1)
residual = ResidualConnection(d_model, dropout=0.1)


x = torch.randn(2, 10, d_model)

# Without residual
output_no_residual = ffn(x)

# With residual
output_with_residual = residual(x, ffn)

print("Input shape:", x.shape)
print("Output (no residual) shape:", output_no_residual.shape)
print("Output (with residual) shape:", output_with_residual.shape)

In [ ]:
# Visualizing Residual Connection Effect

# Creating a sequence of layers WITHOUT residuals
class DeepNetworkNoResidual(nn.Module):
    def __init__(self, d_model, num_layers=6):
        super().__init__()
        self.layers = nn.ModuleList([
            FeedForward(d_model, d_model * 4) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer in self.layers:
            x = layer(x)
        return x

# Create a sequence of layers WITH residuals
class DeepNetworkWithResidual(nn.Module):
    def __init__(self, d_model, num_layers=6):
        super().__init__()
        self.layers = nn.ModuleList([
            FeedForward(d_model, d_model * 4) for _ in range(num_layers)
        ])
        self.residuals = nn.ModuleList([
            ResidualConnection(d_model) for _ in range(num_layers)
        ])

    def forward(self, x):
        for layer, residual in zip(self.layers, self.residuals):
            x = residual(x, layer)
        return x

# Test both
d_model = 128  # Smaller for faster computation
x = torch.randn(1, 10, d_model, requires_grad=True)

net_no_res = DeepNetworkNoResidual(d_model, num_layers=6)
net_with_res = DeepNetworkWithResidual(d_model, num_layers=6)

# Forward pass
output_no_res = net_no_res(x)
output_with_res = net_with_res(x)

# Backward pass (compute gradients)
# NOTE: output.sum() is NOT a valid probe here. After LayerNorm (gamma=1, beta=0)
# every row sums to exactly 0, so its gradient is identically zero - which is why
# the earlier version printed 0.000000 "with residuals". Project onto a fixed
# random direction instead.
probe = torch.randn_like(output_with_res)
loss_no_res = (output_no_res * probe).sum()
loss_with_res = (output_with_res * probe).sum()

loss_no_res.backward()
grad_no_res = x.grad.clone()

x.grad.zero_()
loss_with_res.backward()
grad_with_res = x.grad.clone()

print("Gradient magnitudes:")
print(f"  Without residuals: {grad_no_res.abs().mean():.6f}")
print(f"  With residuals:    {grad_with_res.abs().mean():.6f}")

## 7. Encoder

In [ ]:
# Encoder Block
class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        # Sub-layer 1: Multi-head self-attention
        self.self_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # Sub-layer 2: Feed-forward network
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        # Residual connections with layer norm
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)

    def forward(self, x, mask=None):
        # Sub-layer 1: Self-attention with residual connection
        # Self-attention: Q=K=V=x (all from same input)
        x = self.residual1(x, lambda x: self.self_attention(x, x, x, mask)[0])

        # Sub-layer 2: Feed-forward with residual connection
        x = self.residual2(x, self.feed_forward)

        return x

In [ ]:
# Test Encoder Block

d_model = 512
n_heads = 8
d_ff = 2048
batch_size = 2
seq_len = 10


encoder_block = EncoderBlock(d_model, n_heads, d_ff, dropout=0.1)

x = torch.randn(batch_size, seq_len, d_model)

# Forward pass
output = encoder_block(x, mask=None)

print("Input shape:", x.shape)
print("Output shape:", output.shape) # Ensuring shape matches


# Count parameters
total_params = sum(p.numel() for p in encoder_block.parameters())
print(f"\nTotal parameters in one encoder block: {total_params:,}")
print(f"With 6 blocks: {total_params * 6:,} parameters")

In [ ]:
# Complete encoder (stacked blocks)
class Encoder(nn.Module):
    def __init__(self, num_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        # Stack of encoder blocks
        self.layers = nn.ModuleList([
            EncoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # No extra final LayerNorm: with Post-LN each block already ends in one.

    def forward(self, x, mask=None):

        # Pass through each encoder block
        for layer in self.layers:
            x = layer(x, mask)

        return x

In [ ]:
# Test Encoder

num_layers = 6
encoder = Encoder(num_layers, d_model, n_heads, d_ff, dropout=0.1)

x = torch.randn(batch_size, seq_len, d_model)  # (this would normally come from embedding layer)

# Forward pass
encoded = encoder(x, mask=None)

print("Input shape:", x.shape)
print("Encoded output shape:", encoded.shape)


# Count total parameters
total_params = sum(p.numel() for p in encoder.parameters())
print(f"\nTotal encoder parameters: {total_params:,}")

In [ ]:
# Visualizing Encoder Flow


# simple encoder with fewer layers for visualization
mini_encoder = Encoder(num_layers=2, d_model=128, n_heads=4, d_ff=512, dropout=0.0)
mini_encoder.eval()

x_vis = torch.randn(1, 5, 128)

# Track outputs at each layer
outputs = [x_vis]

with torch.no_grad():
    x = x_vis
    for i, layer in enumerate(mini_encoder.layers):
        x = layer(x, mask=None)
        outputs.append(x.clone())

# Visualize
fig, axes = plt.subplots(1, len(outputs), figsize=(15, 3))

for i, (ax, output) in enumerate(zip(axes, outputs)):
    ax.imshow(output[0].numpy(), aspect='auto', cmap='RdBu')
    if i == 0:
        ax.set_title('Input')
    else:
        ax.set_title(f'After Block {i}')
    ax.set_xlabel('Dimension')
    ax.set_ylabel('Position')

plt.suptitle('Data Flow Through Encoder Blocks')
plt.tight_layout()
plt.show()

## 8. Decoder

In [ ]:
# Single decoder block
class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()

        # Sub-layer 1: Masked self-attention
        self.masked_self_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # Sub-layer 2: Cross-attention (attend to encoder output)
        self.cross_attention = MultiHeadAttention(d_model, n_heads, dropout)

        # Sub-layer 3: Feed-forward network
        self.feed_forward = FeedForward(d_model, d_ff, dropout)

        # Three residual connections with layer norm
        self.residual1 = ResidualConnection(d_model, dropout)
        self.residual2 = ResidualConnection(d_model, dropout)
        self.residual3 = ResidualConnection(d_model, dropout)

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):
        # Sub-layer 1: Masked self-attention (Q=K=V=x, but with mask to prevent looking ahead)

        x = self.residual1(x, lambda x: self.masked_self_attention(x, x, x, tgt_mask)[0])

        # Sub-layer 2: Cross-attention with encoder output (Q from decoder, K and V from encoder)

        x = self.residual2(x, lambda x: self.cross_attention(x, encoder_output, encoder_output, src_mask)[0])

        # Sub-layer 3: Feed-forward
        x = self.residual3(x, self.feed_forward)

        return x

In [ ]:
# Masking functions for the Transformer


def create_look_ahead_mask(size):

    mask = torch.tril(torch.ones(size, size, dtype=torch.bool))  # bool, so `&` works
    return mask  # 1 = attend, 0 = mask


def create_padding_mask(seq, pad_idx=0):

    # Create mask where pad_idx positions are 0, others are 1
    mask = (seq != pad_idx).unsqueeze(1).unsqueeze(2)
    return mask  # (batch_size, 1, 1, seq_len)


def create_target_mask(tgt, pad_idx=0):
    tgt_len = tgt.size(1)

    # Look-ahead mask
    look_ahead_mask = create_look_ahead_mask(tgt_len)
    look_ahead_mask = look_ahead_mask.to(tgt.device)

    # Padding mask
    padding_mask = create_padding_mask(tgt, pad_idx)

    # Combine both masks
    # Broadcasting: (tgt_len, tgt_len) & (batch, 1, 1, tgt_len)
    combined_mask = look_ahead_mask.view(1, 1, tgt_len, tgt_len) & padding_mask  # (batch,1,T,T)

    return combined_mask

In [ ]:
# Test decoder block
d_model = 512
n_heads = 8
d_ff = 2048
batch_size = 2
src_seq_len = 10  # Source (encoder) sequence length
tgt_seq_len = 8   # Target (decoder) sequence length

# Creating decoder block
decoder_block = DecoderBlock(d_model, n_heads, d_ff, dropout=0.1)

# inputs
decoder_input = torch.randn(batch_size, tgt_seq_len, d_model)
encoder_output = torch.randn(batch_size, src_seq_len, d_model)

# Create masks
tgt_mask = create_look_ahead_mask(tgt_seq_len).unsqueeze(0)  # Look-ahead mask

# Forward pass
output = decoder_block(decoder_input, encoder_output, src_mask=None, tgt_mask=tgt_mask)

print("Decoder input shape:", decoder_input.shape)
print("Encoder output shape:", encoder_output.shape)
print("Decoder output shape:", output.shape)


# Count parameters
total_params = sum(p.numel() for p in decoder_block.parameters())
print(f"\nTotal parameters in one decoder block: {total_params:,}")
print(f"Compare to encoder block: Decoder has extra attention layer!")

In [ ]:
# Complete decoder (stack of blocks)
class Decoder(nn.Module):
    def __init__(self, num_layers, d_model, n_heads, d_ff, dropout=0.1):
        super().__init__()


        self.layers = nn.ModuleList([
            DecoderBlock(d_model, n_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])

        # No extra final LayerNorm (Post-LN), as in the encoder.

    def forward(self, x, encoder_output, src_mask=None, tgt_mask=None):

        # Pass through each decoder block
        for layer in self.layers:
            x = layer(x, encoder_output, src_mask, tgt_mask)

        return x

In [ ]:
# Test complete decoder

num_layers = 6
decoder = Decoder(num_layers, d_model, n_heads, d_ff, dropout=0.1)


decoder_input = torch.randn(batch_size, tgt_seq_len, d_model)
encoder_output = torch.randn(batch_size, src_seq_len, d_model)
tgt_mask = create_look_ahead_mask(tgt_seq_len).unsqueeze(0)

# Forward pass
decoded = decoder(decoder_input, encoder_output, src_mask=None, tgt_mask=tgt_mask)

print("Decoder input shape:", decoder_input.shape)
print("Encoder output shape:", encoder_output.shape)
print("Decoded output shape:", decoded.shape)


# Count total parameters
total_params = sum(p.numel() for p in decoder.parameters())
print(f"\nTotal decoder parameters: {total_params:,}")

In [ ]:
# Cross attention visualisation

# a smaller decoder for visualization
mini_decoder = Decoder(num_layers=1, d_model=128, n_heads=4, d_ff=512, dropout=0.0)
mini_decoder.eval()

# inputs
src_len = 10  # Source sequence length
tgt_len = 8   # Target sequence length

decoder_input = torch.randn(1, tgt_len, 128)
encoder_output = torch.randn(1, src_len, 128)
tgt_mask = create_look_ahead_mask(tgt_len).unsqueeze(0)

# Forward pass and capture attention
with torch.no_grad():
    # Get the first decoder block
    decoder_block = mini_decoder.layers[0]

    # Self-attention (masked)
    x_after_self_attn = decoder_block.residual1(
        decoder_input,
        lambda x: decoder_block.masked_self_attention(x, x, x, tgt_mask)[0]
    )

    # Cross-attention
    _, cross_attn_weights = decoder_block.cross_attention(
        x_after_self_attn, encoder_output, encoder_output, None
    )

# Visualizing cross-attention weights for first 2 heads
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for i in range(2):
    ax = axes[i]

    attn = cross_attn_weights[0, i].numpy()

    sns.heatmap(attn, ax=ax, cmap='YlOrRd',
                xticklabels=False, yticklabels=False)
    ax.set_title(f'Cross-Attention Head {i+1}')
    ax.set_xlabel('Source Position (Encoder)')
    ax.set_ylabel('Target Position (Decoder)')

plt.suptitle('How Decoder Attends to Encoder Output', fontsize=14)
plt.tight_layout()
plt.show()

In [ ]:
# Comparing self-attention to cross-attention

with torch.no_grad():
    # Self-attention (masked)
    _, self_attn_weights = decoder_block.masked_self_attention(
        decoder_input, decoder_input, decoder_input, tgt_mask
    )

    # Cross-attention
    _, cross_attn_weights = decoder_block.cross_attention(
        x_after_self_attn, encoder_output, encoder_output, None
    )


fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Self-attention (first head)
axes[0].imshow(self_attn_weights[0, 0].numpy(), cmap='YlOrRd', aspect='auto')
axes[0].set_title('Masked Self-Attention\n(Decoder attending to itself)')
axes[0].set_xlabel('Key Position (Decoder)')
axes[0].set_ylabel('Query Position (Decoder)')

# Cross-attention (first head)
axes[1].imshow(cross_attn_weights[0, 0].numpy(), cmap='YlOrRd', aspect='auto')
axes[1].set_title('Cross-Attention\n(Decoder attending to Encoder)')
axes[1].set_xlabel('Key Position (Encoder)')
axes[1].set_ylabel('Query Position (Decoder)')

plt.tight_layout()
plt.show()

## 9. Full model

In [ ]:
# Complete transformer architecture

class Transformer(nn.Module):
    def __init__(
        self,
        src_vocab_size,
        tgt_vocab_size,
        d_model=512,
        n_heads=8,
        num_encoder_layers=6,
        num_decoder_layers=6,
        d_ff=2048,
        max_seq_len=5000,
        dropout=0.1
    ):
        super().__init__()

        self.d_model = d_model


        self.src_embedding = TransformerEmbedding(src_vocab_size, d_model, max_seq_len, dropout)
        self.tgt_embedding = TransformerEmbedding(tgt_vocab_size, d_model, max_seq_len, dropout)


        self.encoder = Encoder(num_encoder_layers, d_model, n_heads, d_ff, dropout)
        self.decoder = Decoder(num_decoder_layers, d_model, n_heads, d_ff, dropout)

        # Final linear layer to project to vocabulary
        self.fc_out = nn.Linear(d_model, tgt_vocab_size)


        self._init_parameters()

    def _init_parameters(self):
        for p in self.parameters():
            if p.dim() > 1:
                nn.init.xavier_uniform_(p)

    def encode(self, src, src_mask=None):
        src_embedded = self.src_embedding(src)
        return self.encoder(src_embedded, src_mask)

    def decode(self, tgt, encoder_output, src_mask=None, tgt_mask=None):
        tgt_embedded = self.tgt_embedding(tgt)
        return self.decoder(tgt_embedded, encoder_output, src_mask, tgt_mask)

    def forward(self, src, tgt, src_mask=None, tgt_mask=None):

        encoder_output = self.encode(src, src_mask)


        decoder_output = self.decode(tgt, encoder_output, src_mask, tgt_mask)


        output = self.fc_out(decoder_output)

        return output

In [ ]:
# Testing transformer model

src_vocab_size = 10000
tgt_vocab_size = 10000
d_model = 512
n_heads = 8
num_layers = 6
d_ff = 2048

model = Transformer(
    src_vocab_size=src_vocab_size,
    tgt_vocab_size=tgt_vocab_size,
    d_model=d_model,
    n_heads=n_heads,
    num_encoder_layers=num_layers,
    num_decoder_layers=num_layers,
    d_ff=d_ff,
    dropout=0.1
)

# random data
batch_size = 2
src_seq_len = 10
tgt_seq_len = 8

# Random token indices
src = torch.randint(0, src_vocab_size, (batch_size, src_seq_len))
tgt = torch.randint(0, tgt_vocab_size, (batch_size, tgt_seq_len))


tgt_mask = create_look_ahead_mask(tgt_seq_len).unsqueeze(0)


output = model(src, tgt, src_mask=None, tgt_mask=tgt_mask)

print("Source shape:", src.shape)
print("Target shape:", tgt.shape)
print("Output shape:", output.shape)
print(f"Expected: (batch_size={batch_size}, tgt_seq_len={tgt_seq_len}, tgt_vocab_size={tgt_vocab_size})")


total_params = sum(p.numel() for p in model.parameters())
print(f"\nTotal parameters: {total_params:,}")
print(f"That's approximately {total_params / 1e6:.1f}M parameters")

## 10. Sanity-check training: copy task (teacher-forced)

In [ ]:
"""
TASK 1: SEQUENCE COPYING

Input:  [1, 2, 3, 4, 5]
Output: [1, 2, 3, 4, 5]

This tests if the model can:
- Learn simple mapping
- Pass gradients correctly
- Basic encoder-decoder communication
"""

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np

class CopyTaskDataset(Dataset):  # Generates random sequences and their copies
    def __init__(self, num_samples, seq_len, vocab_size):
        self.num_samples = num_samples
        self.seq_len = seq_len
        self.vocab_size = vocab_size

        # Special tokens
        self.PAD_IDX = 0
        self.SOS_IDX = 1  # Start of sequence
        self.EOS_IDX = 2  # End of sequence


        self.data = self._generate_data()

    def _generate_data(self):   #Generate random sequences

        data = []

        for _ in range(self.num_samples):
            seq = torch.randint(3, self.vocab_size, (self.seq_len,))
            data.append(seq)

        return data

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):

        seq = self.data[idx]
        src = seq.clone()

        # Target input: [SOS] + sequence
        tgt_input = torch.cat([torch.tensor([self.SOS_IDX]), seq])

        # Target output: sequence + [EOS]
        tgt_output = torch.cat([seq, torch.tensor([self.EOS_IDX])])

        return src, tgt_input, tgt_output


# Generate dataset
num_samples = 10000
seq_len = 5
vocab_size = 20  # Small vocabulary for testing

train_dataset = CopyTaskDataset(num_samples, seq_len, vocab_size)
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

# Test the dataset
src, tgt_in, tgt_out = train_dataset[0]
print("Example from dataset:")
print(f"Source:          {src.tolist()}")
print(f"Target input:    {tgt_in.tolist()} (with SOS)")
print(f"Target output:   {tgt_out.tolist()} (with EOS)")
print(f"\nDataset size: {len(train_dataset)} samples")
print(f"Batches: {len(train_loader)}")

In [ ]:
"""
Create a smaller Transformer for faster training on this simple task
"""

# Hyperparameters (smaller than paper for faster training)
config = {
    'src_vocab_size': vocab_size,
    'tgt_vocab_size': vocab_size,
    'd_model': 128,           # Reduced from 512
    'n_heads': 4,             # Reduced from 8
    'num_encoder_layers': 2,  # Reduced from 6
    'num_decoder_layers': 2,  # Reduced from 6
    'd_ff': 512,              # Reduced from 2048
    'max_seq_len': 100,
    'dropout': 0.1
}

model = Transformer(**config)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

print(f"Model created and moved to: {device}")
print(f"\nModel parameters: {sum(p.numel() for p in model.parameters()):,}")

print("\nConfiguration:")
for key, value in config.items():
    print(f"  {key}: {value}")

In [ ]:
class TransformerLRScheduler:
    """
    Learning rate scheduler from the paper.
    """

    def __init__(self, optimizer, d_model, warmup_steps=4000):
        self.optimizer = optimizer
        self.d_model = d_model
        self.warmup_steps = warmup_steps
        self.step_num = 0
        self._set_lr(self._lr_at(1))  # avoid lr=0 / 0**-0.5 before the first step

    def _set_lr(self, lr):
        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def _lr_at(self, step):
        step = max(step, 1)
        return (self.d_model ** -0.5) * min(step ** -0.5, step * (self.warmup_steps ** -1.5))

    def step(self):
        self.step_num += 1
        lr = self._get_lr()

        for param_group in self.optimizer.param_groups:
            param_group['lr'] = lr

    def _get_lr(self):
        step = self.step_num


        lr = (self.d_model ** -0.5) * min(
            step ** -0.5,
            step * (self.warmup_steps ** -1.5)
        )

        return lr

    def get_last_lr(self):
        """Get current learning rate"""
        return self._get_lr()


# Visualize the learning rate schedule
steps = np.arange(1, 10000)
d_model = config['d_model']
warmup_steps = 400  # Reduced for smaller model

lrs = []
for step in steps:
    lr = (d_model ** -0.5) * min(step ** -0.5, step * (warmup_steps ** -1.5))
    lrs.append(lr)

plt.figure(figsize=(10, 5))
plt.plot(steps, lrs)
plt.axvline(warmup_steps, color='red', linestyle='--', label=f'Warmup steps ({warmup_steps})')
plt.xlabel('Training Step')
plt.ylabel('Learning Rate')
plt.title('Learning Rate Schedule from Paper')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

print(f"Learning rate schedule:")
print(f"  Warmup steps: {warmup_steps}")
print(f"  Peak LR: {max(lrs):.6f}")
print(f"  LR at step 1000: {lrs[999]:.6f}")

In [ ]:
# Training loop

from tqdm.notebook import tqdm
import time

def train_epoch(model, train_loader, criterion, optimizer, scheduler, device):
    """Train for one epoch"""
    model.train()
    total_loss = 0

    progress_bar = tqdm(train_loader, desc='Training')

    for batch_idx, (src, tgt_input, tgt_output) in enumerate(progress_bar):
        # Move to device
        src = src.to(device)
        tgt_input = tgt_input.to(device)
        tgt_output = tgt_output.to(device)


        tgt_mask = create_look_ahead_mask(tgt_input.size(1)).to(device)
        tgt_mask = tgt_mask.unsqueeze(0)  # Add batch dimension


        optimizer.zero_grad()
        output = model(src, tgt_input, src_mask=None, tgt_mask=tgt_mask)

        # Calculate loss
        # Reshape for cross entropy: (batch * seq_len, vocab_size)
        output = output.reshape(-1, output.size(-1))
        tgt_output = tgt_output.reshape(-1)

        loss = criterion(output, tgt_output)

        # Backward pass
        loss.backward()

        # Clip gradients to prevent explosion
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)

        optimizer.step()
        scheduler.step()

        # Track loss
        total_loss += loss.item()

        # Update progress bar
        progress_bar.set_postfix({
            'loss': f'{loss.item():.4f}',
            'lr': f'{scheduler.get_last_lr():.6f}'
        })

    return total_loss / len(train_loader)


def evaluate(model, data_loader, criterion, device):
    model.eval()
    total_loss = 0
    correct_sequences = 0
    total_sequences = 0

    with torch.no_grad():
        for src, tgt_input, tgt_output in data_loader:
            src = src.to(device)
            tgt_input = tgt_input.to(device)
            tgt_output = tgt_output.to(device)


            tgt_mask = create_look_ahead_mask(tgt_input.size(1)).to(device)
            tgt_mask = tgt_mask.unsqueeze(0)


            output = model(src, tgt_input, src_mask=None, tgt_mask=tgt_mask)


            output_flat = output.reshape(-1, output.size(-1))
            tgt_flat = tgt_output.reshape(-1)
            loss = criterion(output_flat, tgt_flat)
            total_loss += loss.item()


            predictions = output.argmax(dim=-1)
            correct_sequences += (predictions == tgt_output).all(dim=1).sum().item()
            total_sequences += src.size(0)

    avg_loss = total_loss / len(data_loader)
    accuracy = correct_sequences / total_sequences

    return avg_loss, accuracy

In [ ]:
# Training model on copy task

optimizer = torch.optim.Adam(model.parameters(), lr=0, betas=(0.9, 0.98), eps=1e-9)
scheduler = TransformerLRScheduler(optimizer, d_model=config['d_model'], warmup_steps=400)

# Loss function (ignore padding index)
criterion = nn.CrossEntropyLoss(ignore_index=0)

# Training parameters
num_epochs = 20

# Creating validation set
val_dataset = CopyTaskDataset(1000, seq_len, vocab_size)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)


train_losses = []
val_losses = []
val_accuracies = []

print("="*70)
print("TRAINING SEQUENCE COPY TASK")
print("="*70)
print(f"Task: Copy sequences of length {seq_len}")
print(f"Vocabulary size: {vocab_size}")
print(f"Training samples: {len(train_dataset)}")
print(f"Validation samples: {len(val_dataset)}")
print(f"Epochs: {num_epochs}")
print("="*70)
print()

# NOTE: `val_accuracy` here is TEACHER-FORCED exact match on the same set used
# for early stopping. The real (autoregressive, held-out) numbers come from
# transformer/train.py - see README.
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    print("-"*70)

    # Train
    train_loss = train_epoch(model, train_loader, criterion, optimizer, scheduler, device)

    # Evaluate
    val_loss, val_accuracy = evaluate(model, val_loader, criterion, device)

    # Save metrics
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    val_accuracies.append(val_accuracy)

    # Print results
    print(f"\nResults:")
    print(f"  Train Loss: {train_loss:.4f}")
    print(f"  Val Loss:   {val_loss:.4f}")
    print(f"  Val Acc:    {val_accuracy*100:.2f}%")

    # Early stopping if perfect
    if val_accuracy > 0.99:
        print(f"\nAchieved >99% accuracy! Stopping early.")
        break

print("\n" + "="*70)
print("TRAINING COMPLETE!")
print("="*70)

In [ ]:
# Visualise training

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Plot losses
axes[0].plot(train_losses, label='Train Loss', marker='o')
axes[0].plot(val_losses, label='Val Loss', marker='s')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training and Validation Loss')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Plot accuracy
axes[1].plot(val_accuracies, label='Val Accuracy', marker='o', color='green')
axes[1].set_xlabel('Epoch')
axes[1].set_ylabel('Accuracy')
axes[1].set_title('Validation Accuracy')
axes[1].axhline(y=1.0, color='r', linestyle='--', label='Perfect (100%)')
axes[1].legend()
axes[1].grid(True, alpha=0.3)
axes[1].set_ylim([0, 1.05])

plt.tight_layout()
plt.show()

print(f"\nFinal Results:")
print(f"  Best Val Accuracy: {max(val_accuracies)*100:.2f}%")
print(f"  Final Train Loss: {train_losses[-1]:.4f}")
print(f"  Final Val Loss: {val_losses[-1]:.4f}")

In [ ]:
# Testing model (inference)

def predict_sequence(model, src_seq, max_len=20, device='cuda'):

    model.eval()

    # Add batch dimension
    src = src_seq.unsqueeze(0).to(device)

    with torch.no_grad():
        encoder_output = model.encode(src, src_mask=None)

    tgt = torch.tensor([[1]]).to(device)  # SOS_IDX = 1

    for _ in range(max_len):
        tgt_mask = create_look_ahead_mask(tgt.size(1)).to(device)
        tgt_mask = tgt_mask.unsqueeze(0)

        with torch.no_grad():
            output = model.decode(tgt, encoder_output, src_mask=None, tgt_mask=tgt_mask)
            logits = model.fc_out(output)

        # Get next token (greedy decoding)
        next_token = logits[:, -1, :].argmax(dim=-1)

        # Stop if EOS token
        if next_token.item() == 2:  # EOS_IDX = 2
            break

        # Append to sequence
        tgt = torch.cat([tgt, next_token.unsqueeze(0)], dim=1)

    return tgt.squeeze(0)[1:]  # Remove SOS token


print("="*70)
print("TESTING THE MODEL")
print("="*70)
print()

num_test_examples = 10
test_dataset = CopyTaskDataset(num_test_examples, seq_len, vocab_size)

correct = 0
for i in range(num_test_examples):
    src, _, tgt_expected = test_dataset[i]

    prediction = predict_sequence(model, src, max_len=seq_len+2, device=device)

    tgt_expected = tgt_expected[:-1]  # Remove EOS

    is_correct = torch.equal(prediction.cpu(), tgt_expected)
    correct += is_correct

    # Print example
    status = "✅" if is_correct else "❌"
    print(f"Example {i+1}: {status}")
    print(f"  Input:    {src.tolist()}")
    print(f"  Expected: {tgt_expected.tolist()}")
    print(f"  Got:      {prediction.cpu().tolist()}")
    print()

accuracy = correct / num_test_examples
print("="*70)
print(f"Test Accuracy: {accuracy*100:.1f}% ({correct}/{num_test_examples})")
print("="*70)

if accuracy == 1.0:
    print("\n PERFECT! Model can copy sequences flawlessly!")
elif accuracy > 0.8:
    print("\n Good! Model is mostly working")
else:
    print("\n Needs more training or debugging")